## DM Project: Impact of Weather on US Flight Delays
This notebook integrates the cleaned flight dataset with the cleaned weather dataset.
- Integration method: Left join on FL_DATE and ORIGIN
- Enrichment: 3 new weather columns added to each flight record
- Integration errors are measured and documented

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

# Load both cleaned datasets
flights_df = pd.read_csv('/content/drive/MyDrive/DM_Project/Data/Cleaned/flights_cleaned.csv')
weather_df = pd.read_csv('/content/drive/MyDrive/DM_Project/Data/Cleaned/weather_cleaned.csv')

# Fix date formats
flights_df['FL_DATE'] = pd.to_datetime(flights_df['FL_DATE'])
weather_df['FL_DATE'] = pd.to_datetime(weather_df['FL_DATE'])

print("Flights loaded:", flights_df.shape)
print("Weather loaded:", weather_df.shape)

Flights loaded: (2919079, 32)
Weather loaded: (1495, 5)


In [ ]:
top_airports = flights_df['ORIGIN'].value_counts().head(10).index.tolist()
df_top = flights_df[flights_df['ORIGIN'].isin(top_airports)]
df_sample = df_top[df_top['FL_DATE'].dt.month == 6]
print("Filtered dataset:", df_sample.shape)
print("Airports:", top_airports)

Filtered dataset: (84602, 32)
Airports: ['ATL', 'DFW', 'ORD', 'DEN', 'CLT', 'LAX', 'PHX', 'LAS', 'SEA', 'MCO']


In [ ]:
# Left join on FL_DATE and ORIGIN
integrated_df = df_sample.merge(weather_df, on=['FL_DATE', 'ORIGIN'], how='left')
print("Integration done!")
print("Shape after integration:", integrated_df.shape)
print("\nNew columns added:", ['temperature_max', 'precipitation_mm', 'windspeed_max_kmh'])

Integration done!
Shape after integration: (84602, 35)

New columns added: ['temperature_max', 'precipitation_mm', 'windspeed_max_kmh']


In [ ]:
print("=== INTEGRATION ERROR MEASUREMENT ===")

# Missing weather records after join
missing_weather = integrated_df['temperature_max'].isnull().sum()
total = integrated_df.shape[0]
match_rate = round((1 - missing_weather/total) * 100, 2)

print(f"Total flight records: {total}")
print(f"Successfully enriched: {total - missing_weather} ({match_rate}%)")
print(f"Missing weather data: {missing_weather} ({round(missing_weather/total*100, 2)}%)")

# Drop unmatched rows
integrated_df = integrated_df.dropna(subset=['temperature_max', 'precipitation_mm', 'windspeed_max_kmh'])
print(f"\nFinal integrated records: {integrated_df.shape[0]}")

=== INTEGRATION ERROR MEASUREMENT ===
Total flight records: 84602
Successfully enriched: 84356 (99.71%)
Missing weather data: 246 (0.29%)

Final integrated records: 84356


In [ ]:
print("=== SAMPLE OF INTEGRATED DATA ===")
print(integrated_df[['FL_DATE', 'ORIGIN', 'DEP_DELAY', 'DELAY_DUE_WEATHER',
                       'temperature_max', 'precipitation_mm',
                       'windspeed_max_kmh']].head(10))

print("\nCommon columns used for join: FL_DATE, ORIGIN")
print("Columns before integration:", df_sample.shape[1])
print("Columns after integration:", integrated_df.shape[1])
print("New columns added:", integrated_df.shape[1] - df_sample.shape[1])

=== SAMPLE OF INTEGRATED DATA ===
     FL_DATE ORIGIN  DEP_DELAY  DELAY_DUE_WEATHER  temperature_max  \
0 2021-06-11    ATL       69.0                0.0             27.5   
1 2021-06-08    ORD        1.0                0.0             29.4   
2 2021-06-01    DFW       -1.0                0.0             24.4   
3 2021-06-26    SEA       22.0                0.0             34.5   
4 2022-06-10    PHX       23.0                0.0             44.5   
5 2019-06-10    SEA       -7.0                0.0             22.3   
6 2021-06-17    LAX       -6.0                0.0             27.0   
7 2019-06-06    ORD       -7.0                0.0             19.7   
8 2021-06-13    SEA       -1.0                0.0             18.5   
9 2021-06-03    DEN      104.0                0.0             26.0   

   precipitation_mm  windspeed_max_kmh  
0              10.4               19.3  
1               6.0                9.3  
2               0.9               19.4  
3               0.0            

In [ ]:
integrated_df.to_csv('/content/drive/MyDrive/DM_Project/Data/Integrated/flights_weather_integrated.csv', index=False)
print("flights_weather_integrated.csv saved to Data/Integrated/")

flights_weather_integrated.csv saved to Data/Integrated/
